# Init

In [0]:
# importing liabraries
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import *
from pyspark.sql import Window

# Rename config

In [0]:
# Rename config
RENAME_MAP = {
    "sls_ord_num": "sales_order_number",
    "sls_prd_key": "sales_product_key",
    "sls_cust_id": "sales_customer_id",
    "sls_order_dt": "sales_order_date",
    "sls_ship_dt": "sales_shiping_date",
    "sls_due_dt": "sales_due_date",
    "sls_sales": "total_sales",
    "sls_quantity": "sales_quantity",
    "sls_price": "sales_price"
}

# Read from Bronze

In [0]:
# read spark table
df = spark.table("workspace.bronze.crm_sales_details_raw")

# Transformation

## Rename columns

In [0]:
# column Rename
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Trim

In [0]:
# string data triming
for field in df.schema.fields:
    if field.dataType == StringType():
        df = df.withColumn(field.name, trim(col(field.name)))

## Int to String to Date

In [0]:
from pyspark.sql import functions as F

date_columns = ["sales_order_date", "sales_shiping_date", "sales_due_date"]

# df_cleaned = df

# Loop through and apply the exact same transformation rules dynamically
for col_name in date_columns:
    col_str = F.col(col_name).cast("string")
    
    df = df.withColumn(
        col_name,
        F.when((F.col(col_name) == 0) | (F.length(col_str) != 8), F.lit(None))
         .otherwise(F.to_date(col_str, "yyyyMMdd"))
    )


## Recalculate total_sales and sales_price value

In [0]:
from pyspark.sql import functions as F

# ==============================================================================
# STEP 1: FIX TOTAL SALES FIRST
# Enforces positive values and recalculates sales if they mismatch quantity * price
# ==============================================================================
df = df.withColumn(
    "total_sales",
    F.when(
        (F.col("total_sales") != (F.col("sales_quantity") * F.abs(F.col("sales_price")))) | 
        (F.col("total_sales").isNull()) | 
        (F.col("total_sales") <= 0),
        F.abs(F.col("sales_quantity") * F.col("sales_price")) # Recalculate cleanly
    ).otherwise(F.abs(F.col("total_sales"))) # Ensure existing sales are positive
)

# ==============================================================================
# STEP 2: FIX UNIT PRICES USING TREATED SALES DATA
# Fixes nulls, negatives, and mathematical discrepancies by back-calculating price
# ==============================================================================
df = df.withColumn(
    "sales_price",
    F.when(
        (F.col("sales_price").isNull()) | 
        (F.col("sales_price") <= 0) |
        (F.col("total_sales") != (F.col("sales_quantity") * F.col("sales_price"))),
        
        # Safe division: handles division-by-zero if quantity is 0
        F.when(F.col("sales_quantity") == 0, F.lit(0.00))
         .otherwise(F.col("total_sales") / F.col("sales_quantity"))
    ).otherwise(F.col("sales_price"))
).withColumn(
    # Final step: cast columns to standard financial decimal formats
    "total_sales", F.col("total_sales").cast("decimal(10,2)")
).withColumn(
    "sales_price", F.col("sales_price").cast("decimal(10,2)")
)


## checking validity

In [0]:
from pyspark.sql import functions as F

# 1. Define the violation conditions using the new clean column names
is_invalid_sales = (
    (F.col("total_sales") != (F.col("sales_quantity") * F.abs(F.col("sales_price")))) | 
    (F.col("total_sales").isNull()) | 
    (F.col("total_sales") <= 0)
)

is_invalid_price = (
    (F.col("sales_price").isNull()) | 
    (F.col("sales_price") <= 0)
)

# 2. Filter the renamed DataFrame to isolate bad rows
# Note: replace 'df' with whatever variable name holds your renamed DataFrame
df_violations = df.filter(is_invalid_sales | is_invalid_price)

# 3. Add a diagnostic reason column to label the errors
df_audit = df_violations.withColumn(
    "violation_reason",
    F.concat_ws(
        " | ",
        F.when(is_invalid_sales, F.lit("Bad Sales Amount")).otherwise(F.lit("")),
        F.when(is_invalid_price, F.lit("Bad Unit Price")).otherwise(F.lit(""))
    )
)

# 4. Display the results
print(f"🚨 Found {df_audit.count()} rows violating financial logic:")
display(df_audit.select(
    "sales_order_date", 
    "sales_quantity", 
    "sales_price", 
    "total_sales", 
    "violation_reason"
))


In [0]:
df.display()

# Write to Silver

In [0]:
(
    df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver.crm_sales_details")
)